<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

### RapidFire AI RAG/Context Engineering Tutorial Use Case: SciFact Q&A Chatbot

In [1]:
# Block out access to GPUs to make it a CPU-only machine
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [2]:
import getpass

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")
PINECONE_API_KEY = getpass.getpass("Enter your Pinecone API key: ")
GOOGLE_API_KEY = getpass.getpass("Enter your Google API key: ")

In [3]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFLangChainRagSpec, RFOpenAIAPIModelConfig, RFGeminiAPIModelConfig, RFPromptManager, RFGridSearch
import re, json
from typing import List as listtype, Dict, Any

### Load Dataset and Rename Columns

In [4]:
# Delete pinecone indexes to make space

In [5]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY, source_tag="rapidfireai")

indexes = pc.list_indexes()
len(indexes)

7

In [6]:
for index in indexes:
  if index.name != "fiqa":
    print(f"Deleting {index.name}...")
    pc.delete_index(index.name)

print("Done.")

Deleting rf-5352ff9c2bf3887efc873ee7a067f46674274b75e1...
Deleting rf-de6b64e235c735a04d37a1105076c5d9eb8ee95f26...
Deleting rf-df4da8687a24a1bc197c55a29c43143c7cc4ae3095...
Deleting rf-663c0fe8fd9044959528895a9d330f59deefc94a47...
Deleting rf-4a022f43ed0fdab009c3d205f9051cacc0cf39057b...
Deleting rf-7c5bd6972fe46f529918c4298fb77ce870b930e28e...
Done.


In [7]:
import pandas as pd
from datasets import Dataset
import json, random
from pathlib import Path

# Dataset directory
dataset_dir = Path("datasets")

# Dataset directory is now in tutorial_notebooks/evals/datasets
data = []
with open(str(dataset_dir / "scifact" / "queries.jsonl"), "r") as f:
    for line in f:
        data.append(json.loads(line))

for d in data:
    if d["metadata"]:
        for info in d["metadata"].values():
            tags = set([meta["label"] for meta in info])
            assert len(tags) == 1
            d["label"] = tags.pop()  # SUPPORT or CONTRADICT
    else:
        d["label"] = "NOINFO"

scifact_dataset = {
    "query": [d["text"] for d in data],
    "query_id": [d["_id"] for d in data],
    "label": [d["label"] for d in data],
}
scifact_dataset = Dataset.from_dict(scifact_dataset)

qrels = pd.read_csv(str(dataset_dir / "scifact" / "qrels.tsv"), sep="\t")
qrels = qrels.rename(
    columns={"query-id": "query_id", "corpus-id": "corpus_id", "score": "relevance"}
)

# Downsample queries and corpus JOINTLY
sample_fraction = 0.02  # low sample_fraction makes demo faster but degrades metrics; set to 1.0 for full evals if running on a local machine.
rseed = 1
random.seed(rseed)

# Step 1: Sample queries
sample_size = int(len(scifact_dataset) * sample_fraction)
scifact_dataset = scifact_dataset.shuffle(seed=rseed).select(range(sample_size))

# Convert query_ids to integers for matching
query_ids = set([int(qid) for qid in scifact_dataset["query_id"]])

# Step 2: Get all corpus docs relevant to sampled queries
qrels_filtered = qrels[qrels["query_id"].isin(query_ids)]
relevant_corpus_ids = set(qrels_filtered["corpus_id"].tolist())

print(f"Using {len(scifact_dataset)} queries")
print(f"Found {len(relevant_corpus_ids)} relevant documents for these queries")

# Step 3: Load corpus and filter to relevant docs
input_file = dataset_dir / "scifact" / "corpus.jsonl"
output_file = dataset_dir / "scifact" / "corpus_sampled.jsonl"

with open(input_file, 'r') as f:
    all_corpus = [json.loads(line) for line in f]

# Keep only relevant documents (convert _id to int for matching)
sampled_corpus = [doc for doc in all_corpus if int(doc["_id"]) in relevant_corpus_ids]

# Write sampled corpus
with open(output_file, 'w') as f:
    for doc in sampled_corpus:
        f.write(json.dumps(doc) + '\n')

print(f"Sampled {len(sampled_corpus)} documents from {len(all_corpus)} total")
print(f"Saved to: {output_file}")
print(f"Filtered qrels to {len(qrels_filtered)} relevance judgments")

# Update qrels to match
qrels = qrels_filtered

Using 16 queries
Found 18 relevant documents for these queries
Sampled 18 documents from 5183 total
Saved to: datasets/scifact/corpus_sampled.jsonl
Filtered qrels to 18 relevance judgments


### Create Experiment

In [8]:
experiment = Experiment(experiment_name="exp1-scifact-openai-pinecone", mode="evals")

An experiment with the same name already exists. Created a new experiment 'exp1-scifact-openai-pinecone_13' with Experiment ID: 14 at /home/ubuntu/rapidfireai/rapidfire_experiments/exp1-scifact-openai-pinecone_13
Created directory: /home/ubuntu/rapidfireai/logs/exp1-scifact-openai-pinecone_13
Allocating 60.0 CPUs and 0.0 GPUs to the experiment


### Define Partial Multi-Config Knobs for LangChain part of RAG Pipeline using RapidFire AI Wrapper APIs

In [9]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from typing import Dict
from pinecone import ServerlessSpec

batch_size = 8


def metadata_func(record: Dict, metadata: Dict):
    metadata["corpus_id"] = int(record.get("_id"))
    metadata["title"] = record.get("title")
    return metadata


def custom_template(doc: Document) -> str:
    return f"{doc.metadata['title']}:\n{doc.page_content}"


# CPU-based RAG
rag_cpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path=str(dataset_dir / "scifact"),
        glob="corpus_sampled.jsonl",
        loader_cls=JSONLoader,
        loader_kwargs={
            "jq_schema": ".",
            "content_key": "text",
            "metadata_func": metadata_func,  # store the document id
            "json_lines": True,
            "text_content": False,
        },
        sample_seed=1337,
    ),
    text_splitter=List([
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name="cl100k_base", chunk_size=256, chunk_overlap=32),
        CharacterTextSplitter.from_tiktoken_encoder(encoding_name="gpt2", chunk_size=256, chunk_overlap=0),
        TokenTextSplitter(chunk_size=10, chunk_overlap=0),
    ]), # 3 text splitters
    embedding_cfg=List([
        {
            "class": OpenAIEmbeddings,
            "model": "text-embedding-3-small", # "text-embedding-3-large"]),
            "api_key": OPENAI_API_KEY
        },
        {
            "class": GoogleGenerativeAIEmbeddings,
            "model": "gemini-embedding-001",
            "google_api_key": GOOGLE_API_KEY
        }
    ]), # 2 different embedding models
    vector_store_cfg={
        "type": "pinecone",
        "pinecone_api_key": PINECONE_API_KEY,
        "spec": ServerlessSpec(cloud="gcp", region="us-central1"),
        "metric": "cosine",
        "batch_size": 32,
    }, # Pinecone in create mode
    search_cfg=List([
        {"type": "similarity", "k": 15},
        {"type": "similarity_score_threshold", "k": 10, "score_threshold": 0.5},
        {"type": "mmr", "k": 10}
    ]), # 3 different search types
    reranker_cfg={
          "class": CrossEncoderReranker,
          "model_name": List([
              "cross-encoder/ms-marco-MiniLM-L6-v2",
          ]),
          "top_n": List([2, 3, 5]),
          "model_kwargs": {"device": "cpu"},
    }, # 3 different reranking strategies
    enable_gpu_search=False,
    document_template=custom_template,
) # 54 configs

### Define Data Processing and Postprocessing Functions

In [10]:
INSTRUCTIONS = """
You are a helpful assistant that can verify scientific claims. You will be given a scientific claim and a list of documents that are potentially relevant to the claim. Your job is to determine whether the claim is supported, contradicted, or not addressed by the evidence. You will do so by responding with one of the following options:

- SUPPORT: If the evidence supports the claim.
- CONTRADICT: If the evidence contradicts the claim.
- NOINFO: If the evidence does not provide enough information to determine whether the claim is supported or contradicted.

You will output your final answer after reasoning through the evidence. The final answer should be one of the three options and should be formatted as follows:

Reasoning for the answer #### ANSWER

Here is an example:

claim: High cardiopulmonary fitness causes increased mortality rate.

evidence:
One consequence of inactivity, low cardiorespiratory fitness, is an established risk factor for cardiovascular disease (CVD) morbidity and mortality, but the prevalence of cardiorespiratory fitness has not been quantified in representative US population samples.

Cardiosphere-derived cells transplanted into chick embryos migrated to the truncus arteriosus and cardiac outflow tract and contributed to dorsal root ganglia, spinal nerves, and aortic smooth muscle cells. Lineage studies using double transgenic mice encoding protein 0\u2013Cre/Floxed-EGFP revealed undifferentiated and differentiated neural crest-derived cells in the fetal myocardium

Patients undergoing dialysis have a substantially increased risk of cardiovascular mortality and morbidity. Although several trials have shown the cardiovascular benefits of lowering blood pressure in the general population, there is uncertainty about the efficacy and tolerability of reducing blood pressure in patients on dialysis

Response: The evidence suggests that low cardiorespiratory fitness is a known risk factor for cardiovascular disease and therefore the claim is contradicted. #### CONTRADICT
"""

In [11]:
def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)
    retrieved_documents = [
        [doc.metadata["corpus_id"] for doc in docs] for docs in all_context
    ]
    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"\nClaim:\n{question}. \nEvidence:\n{context}. \nYour response:",
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }

def gemini_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)
    retrieved_documents = [
        [doc.metadata["corpus_id"] for doc in docs] for docs in all_context
    ]
    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "parts": [INSTRUCTIONS]},
                {
                    "role": "user",
                    "parts": [
                        {"text": f"\nClaim:\n{question}. \nEvidence:\n{context}. \nYour response:"}
                    ],
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }

def extract_solution(answer):
    solution = re.search(r"####\s*(SUPPORT|CONTRADICT|NOINFO)", answer, re.IGNORECASE)
    if solution is None:
        return "INVALID"
    return solution.group(1).upper()


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Function to postprocess outputs produced by generator model"""
    # Get ground truth documents for each query; can be done in preprocess_fn too but done here for clarity
    batch["ground_truth_documents"] = [
        qrels[qrels["query_id"] == query_id]["corpus_id"].tolist()
        for query_id in batch["query_id"]
    ]
    batch["answer"] = [extract_solution(answer) for answer in batch["generated_text"]]
    return batch

### Define Custom Eval Metrics Functions

In [12]:
import math


def compute_ndcg_at_k(retrieved_docs: set, expected_docs: set, k=3):
    """Utility function to compute NDCG@k"""
    relevance = [1 if doc in expected_docs else 0 for doc in list(retrieved_docs)[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    # IDCG: perfect ranking limited by min(k, len(expected_docs))
    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [3] * ideal_length + [0] * (k - ideal_length)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0


def compute_rr(retrieved_docs: set, expected_docs: set):
    """Utility function to compute Reciprocal Rank (RR) for a single query"""
    rr = 0
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            rr = 1 / (i + 1)
            break
    return rr

def compute_accuracy(predictions, ground_truth):
    """Label prediction accuracy: SUPPORT, CONTRADICT, NOINFO"""
    return sum(1 for pred, gt in zip(predictions, ground_truth) if pred == gt) / len(predictions)

def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Function to compute all eval metrics based on retrievals and/or generations"""

    true_positives, precisions, recalls, f1_scores, ndcgs, rrs, acc = 0, [], [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        retrieved_set = set(pred[:3])

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(compute_ndcg_at_k(retrieved_set, expected_set, k=3))
        rrs.append(compute_rr(retrieved_set, expected_set))

    accuracy = compute_accuracy(batch["answer"], batch["label"])


    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@3": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
        "Accuracy": {"value": accuracy}
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Function to accumulate eval metrics across all batches"""

    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@3", "MRR", "Accuracy"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                )
                / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

### Define Partial Multi-Config Knobs for OpenAI Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

In [13]:
# 2 openai configs with different sizes of generator models and different reasoning levels
openai_config = RFOpenAIAPIModelConfig(
    client_config={"api_key": OPENAI_API_KEY, "max_retries": 2},
    model_config={
        "model": "gpt-5-nano",
        "max_completion_tokens": 2048,
        "reasoning_effort": "medium",
    },
    rpm_limit=30_000, # Request per minute (RPM) needs to be set based on your account tier and the specific model used
    tpm_limit=180_000_000, # Token per minute (TPM) needs to be set based on your account tier and the specific model used
    rag=rag_cpu,
    prompt_manager=None,
)

gemini_config = RFGeminiAPIModelConfig(
    client_config={"api_key": GOOGLE_API_KEY},
    model_config={
        "model": "gemini-2.5-flash-lite",
        "max_output_tokens": 2048,
    },
    rpm_limit=30_000, # Request per minute (RPM) needs to be set based on your account tier and the specific model used
    tpm_limit=30_000_000, # Token per minute (TPM) needs to be set based on your account tier and the specific model used
    rag=rag_cpu,
    prompt_manager=None,
)


config_set = List([{
        "openai_config": openai_config,
        "batch_size": batch_size,
        "preprocess_fn": openai_sample_preprocess_fn,
        "postprocess_fn": sample_postprocess_fn,
        "compute_metrics_fn": sample_compute_metrics_fn,
        "accumulate_metrics_fn": sample_accumulate_metrics_fn,
        "online_strategy_kwargs": {
            "strategy_name": "normal",
            "confidence_level": 0.95,
            "use_fpc": True,
        },
    },
    {
        "gemini_config": gemini_config,
        "batch_size": batch_size,
        "preprocess_fn": gemini_sample_preprocess_fn,
        "postprocess_fn": sample_postprocess_fn,
        "compute_metrics_fn": sample_compute_metrics_fn,
        "accumulate_metrics_fn": sample_accumulate_metrics_fn,
        "online_strategy_kwargs": {
            "strategy_name": "normal",
            "confidence_level": 0.95,
            "use_fpc": True,
        },
    }
])

### Create Config Group

In [14]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

### Run Multi-Config Evals

In [15]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks

# Available CPU cores are equally divided across all concurrent actors.
results = experiment.run_evals(
    config_group=config_group,
    dataset=scifact_dataset,
    num_shards=2,
    seed=42,
)

=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store,Index Name,Namespace
1,Complete,29.3s,CPU,Pinecone,rf-37c6055b1714546e73e35218955f614dfa6c29ab6e,''
2,Complete,29.3s,CPU,Pinecone,rf-4b8898e24ae3cfbcb45b9a770dcf613064dc8558f1,''
3,Complete,29.3s,CPU,Pinecone,rf-07c3a15bfb391ab99e2d2acf785d13b2be61c05fed,''
4,Complete,29.3s,CPU,Pinecone,rf-caddcb6775a7e30d174e116ad09ca3428d21836288,''
5,Complete,46.2s,CPU,Pinecone,rf-cba12035932a1c0a636dd7d355541381373b591e96,''
6,Complete,46.2s,CPU,Pinecone,rf-97db40a64ee5c2d9494abc928ee154473ce4a0d056,''



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter_cfg.add_start_index,text_splitter_cfg.chunk_overlap,text_splitter_cfg.chunk_size,text_splitter_cfg.keep_separator,text_splitter_cfg.strip_whitespace,text_splitter_cfg.type,embedding_cfg.class,embedding_cfg.model,vector_store_cfg.batch_size,vector_store_cfg.metric,vector_store_cfg.spec,vector_store_cfg.type,search_cfg.fetch_k,search_cfg.k,search_cfg.lambda_mult,search_cfg.score_threshold,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Accuracy,F1 Score,MRR,NDCG@3,Precision,Processing Time,Recall,Samples Per Second,Samples Processed,Throughput,Total
1,gpt-5-nano,COMPLETED,2/2,0.000,False,32,256,True,True,RecursiveCharacterTextSplitter(tiktoken:cl100k_base),OpenAIEmbeddings,text-embedding-3-small,32,cosine,"ServerlessSpec(cloud='gcp', region='us-central1')",pinecone,-,15,-,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,2,"max_completion_tokens=2048, reasoning_effort=medium","68.75% [68.75%, 68.75%]","87.50% [87.50%, 87.50%]","87.50% [87.50%, 87.50%]","0.2945 [0.2945, 0.2945]","84.38% [84.38%, 84.38%]",361.47 seconds,"96.88% [96.88%, 96.88%]",0.04,16,0.1/s,16
2,gpt-5-nano,COMPLETED,2/2,0.000,False,32,256,True,True,RecursiveCharacterTextSplitter(tiktoken:cl100k_base),OpenAIEmbeddings,text-embedding-3-small,32,cosine,"ServerlessSpec(cloud='gcp', region='us-central1')",pinecone,-,15,-,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,3,"max_completion_tokens=2048, reasoning_effort=medium","62.50% [62.50%, 62.50%]","76.88% [76.88%, 76.88%]","81.25% [81.25%, 81.25%]","0.2867 [0.2867, 0.2867]","67.71% [67.71%, 67.71%]",360.90 seconds,"100.00% [100.00%, 100.00%]",0.04,16,0.1/s,16
3,gpt-5-nano,COMPLETED,2/2,0.000,False,32,256,True,True,RecursiveCharacterTextSplitter(tiktoken:cl100k_base),OpenAIEmbeddings,text-embedding-3-small,32,cosine,"ServerlessSpec(cloud='gcp', region='us-central1')",pinecone,-,15,-,-,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=medium","56.25% [56.25%, 56.25%]","76.88% [76.88%, 76.88%]","81.25% [81.25%, 81.25%]","0.2867 [0.2867, 0.2867]","67.71% [67.71%, 67.71%]",360.35 seconds,"100.00% [100.00%, 100.00%]",0.04,16,0.1/s,16
4,gpt-5-nano,COMPLETED,2/2,0.000,False,32,256,True,True,RecursiveCharacterTextSplitter(tiktoken:cl100k_base),OpenAIEmbeddings,text-embedding-3-small,32,cosine,"ServerlessSpec(cloud='gcp', region='us-central1')",pinecone,-,10,-,0.5,similarity_score_threshold,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,2,"max_completion_tokens=2048, reasoning_effort=medium","56.25% [56.25%, 56.25%]","87.50% [87.50%, 87.50%]","87.50% [87.50%, 87.50%]","0.2945 [0.2945, 0.2945]","84.38% [84.38%, 84.38%]",359.79 seconds,"96.88% [96.88%, 96.88%]",0.04,16,0.1/s,16
5,gpt-5-nano,COMPLETED,2/2,0.000,False,32,256,True,True,RecursiveCharacterTextSplitter(tiktoken:cl100k_base),OpenAIEmbeddings,text-embedding-3-small,32,cosine,"ServerlessSpec(cloud='gcp', region='us-central1')",pinecone,-,10,-,0.5,similarity_score_threshold,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,3,"max_completion_tokens=2048, reasoning_effort=medium","43.75% [43.75%, 43.75%]","76.88% [76.88%, 76.88%]","81.25% [81.25%, 81.25%]","0.2867 [0.2867, 0.2867]","67.71% [67.71%, 67.71%]",359.26 seconds,"100.00% [100.00%, 100.00%]",0.04,16,0.1/s,16
6,gpt-5-nano,COMPLETED,2/2,0.000,False,32,256,True,True,RecursiveCharacterTextSplitter(tiktoken:cl100k_base),OpenAIEmbeddings,text-embedding-3-small,32,cosine,"ServerlessSpec(cloud='gcp', region='us-central1')",pinecone,-,10,-,0.5,similarity_score_threshold,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=medium","43.75% [43.75%, 43.75%]","76.88% [76.88%, 76.88%]","81.25% [81.25%, 81.25%]","0.2867 [0.2867, 0.2867]","67.71% [67.71%, 67.71%]",358.73 seconds,"100.00% [100.00%, 100.00%]",0.04,16,0.1/s,16
7,gpt-5-nano,COMPLETED,2/2,0.000,

### View Results

In [16]:
# Convert results dict to DataFrame
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

results_df

,run_id,model_name,text_splitter_cfg,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second,Total,Precision,Recall,F1 Score,NDCG@3,MRR,Accuracy
0,1,gpt-5-nano,{'type': 'RecursiveCharacterTextSplitter(tikto...,"{'class': 'OpenAIEmbeddings', 'model': 'text-e...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name'...","{'max_completion_tokens': 2048, 'reasoning_eff...",16,361.47 seconds,0.04,16,0.843750,0.96875,0.875000,0.294518,0.87500,0.6875
1,2,gpt-5-nano,{'type': 'RecursiveCharacterTextSplitter(tikto...,"{'class': 'OpenAIEmbeddings', 'model': 'text-e...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name'...","{'max_completion_tokens': 2048, 'reasoning_eff...",16,360.90 seconds,0.04,16,0.677083,1.00000,0.768750,0.286705,0.81250,0.6250
2,3,gpt-5-nano,{'type': 'RecursiveCharacterTextSplitter(tikto...,"{'class': 'OpenAIEmbeddings', 'model': 'text-e...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name'...","{'max_completion_tokens': 2048, 'reasoning_eff...",16,360.35 seconds,0.04,16,0.677083,1.00000,0.768750,0.286705,0.81250,0.5625
3,4,gpt-5-nano,{'type': 'RecursiveCharacterTextSplitter(tikto...,"{'class': 'OpenAIEmbeddings', 'model': 'text-e...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 10...","{'class': 'CrossEncoderReranker', 'model_name'...","{'max_completion_tokens': 2048, 'reasoning_eff...",16,359.79 seconds,0.04,16,0.843750,0.96875,0.875000,0.294518,0.87500,0.5625
4,5,gpt-5-nano,{'type': 'RecursiveCharacterTextSplitter(tikto...,"{'class': 'OpenAIEmbeddings', 'model': 'text-e...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 10...","{'class': 'CrossEncoderReranker', 'model_name'...","{'max_completion_tokens': 2048, 'reasoning_eff...",16,359.26 seconds,0.04,16,0.677083,1.00000,0.768750,0.286705,0.81250,0.4375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,104,gemini-2.5-flash-lite,"{'type': 'TokenTextSplitter', 'chunk_size': 10...","{'class': 'GoogleGenerativeAIEmbeddings', 'mod...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 10...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_output_tokens': 2048},16,219.88 seconds,0.07,16,1.000000,0.96875,0.979167,0.325274,1.00000,0.3750
104,105,gemini-2.5-flash-lite,"{'type': 'TokenTextSplitter', 'chunk_size': 10...","{'class': 'GoogleGenerativeAIEmbeddings', 'mod...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'similarity_score_threshold', 'k': 10...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_output_tokens': 2048},16,219.63 seconds,0.07,16,1.000000,0.96875,0.979167,0.325274,1.00000,0.5000
105,106,gemini-2.5-flash-lite,"{'type': 'TokenTextSplitter', 'chunk_size': 10...","{'class': 'GoogleGenerativeAIEmbeddings', 'mod...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_output_tokens': 2048},16,218.01 seconds,0.07,16,0.968750,0.93750,0.937500,0.309526,0.96875,0.4375
106,107,gemini-2.5-flash-lite,"{'type': 'TokenTextSplitter', 'chunk_size': 10...","{'class': 'GoogleGenerativeAIEmbeddings', 'mod...","{'type': 'pinecone', 'pinecone_api_key': 'pcsk...","{'type': 'mmr', 'k': 10, 'fetch_k': 20, 'lambd...","{'class': 'CrossEncoderReranker', 'model_name'...",{'max_output_tokens': 2048},16,216.79 seconds,0.07,16,0.968750,0.96875,0.958333,0.317585,0.96875,0.4375


### End Experiment

In [17]:
experiment.end()

Experiment exp1-scifact-openai-pinecone_13 ended


### View RapidFire AI Log Files

In [18]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

📄 Log File: /home/ubuntu/rapidfireai/logs/exp1-scifact-openai-pinecone_13/rapidfire.log

Last 30 lines of rapidfire.log:
2026-04-01 23:26:08 | Controller | INFO | controller.py:811 | [exp1-scifact-openai-pinecone_13:Controller] Pipeline 100 (Pipeline 100) completed successfully
2026-04-01 23:26:09 | Experiment | INFO | metric_rfmetric_manager.py:190 | [exp1-scifact-openai-pinecone_13:Experiment] Ending run: 09b7b6699193441a9b7063ae4fb89571, 101 in rf_mlflow
2026-04-01 23:26:09 | Experiment | WARNING | metric_mlflow_manager.py:101 | [exp1-scifact-openai-pinecone_13:Experiment] Run 09b7b6699193441a9b7063ae4fb89571 is not the active run, no local context to clear
2026-04-01 23:26:09 | Controller | INFO | controller.py:811 | [exp1-scifact-openai-pinecone_13:Controller] Pipeline 101 (Pipeline 101) completed successfully
2026-04-01 23:26:09 | Experiment | INFO | metric_rfmetric_manager.py:190 | [exp1-scifact-openai-pinecone_13:Experiment] Ending run: 9079199944924b45b24ff3cf865c2001, 102 in 